# Feature tables, comparisons and figures

> **Notebook role:** production-style preparation of analysis matrices and condition-level visual summaries.

## 1. Separate metadata from measured features

Object identifiers, image names, conditions and batches remain untouched while
quality filters operate on the numeric measurement columns.

In [ ]:
from pathlib import Path
import pandas as pd

combined = pd.read_csv(Path("outputs/representations.csv"))
metadata_columns = ["object_id", "file", "condition", "batch", "subbatch"]
metadata_columns = [column for column in metadata_columns if column in combined]
feature_columns = [column for column in combined if column not in metadata_columns]
combined[metadata_columns].head(), combined[feature_columns].describe().T

## 2. Remove uninformative and redundant features

Low-uniqueness and high-correlation filters are applied in a fixed order. The
retained column list becomes part of the analysis record.

In [ ]:
from nuclear_table_tools import drop_high_corr_columns, drop_low_unique_columns

analysis_table = drop_low_unique_columns(combined.copy(), feature_columns, unique_threshold=10)
retained = [column for column in feature_columns if column in analysis_table]
analysis_table = drop_high_corr_columns(analysis_table, retained, corr_threshold=0.95)
retained = [column for column in retained if column in analysis_table]
analysis_table.shape, retained[:10]

## 3. Visualize structure before modelling

Standardized heatmaps expose condition-level patterns, batch structure and
outliers. Embeddings give a complementary object-level view.

## Representative result

![Standardized feature heatmap](../assets/results/screening-heatmap.png)

In [ ]:
import seaborn as sns

matrix = analysis_table.set_index("object_id")[retained]
standardized = (matrix - matrix.mean()) / matrix.std().replace(0, 1)
sns.clustermap(standardized.T, cmap="viridis", center=0)

## 4. Evaluate condition-level predictions

Confusion matrices make adjacent or transitional phenotypes visible rather than
reducing performance to a single accuracy value.

![Condition classification matrix](../assets/results/classification-matrix.png)

In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay, confusion_matrix

predictions = pd.read_csv(Path("outputs/condition_predictions.csv"))
true_condition = predictions["condition"]
predicted_condition = predictions["predicted_condition"]
condition_order = sorted(set(true_condition) | set(predicted_condition))
matrix = confusion_matrix(true_condition, predicted_condition, normalize="true")
ConfusionMatrixDisplay(matrix, display_labels=condition_order).plot(cmap="magma")